In [0]:
dbutils.widgets.removeAll()

dbutils.widgets.text("catalogo", "catalog_tr")
dbutils.widgets.text("esquema_bronze", "bronze")
dbutils.widgets.text("tabla_bronze", "transactions_bronze")
dbutils.widgets.text("tabla_drive", "drive_bronze")
dbutils.widgets.text("esquema_silver", "silver")
dbutils.widgets.text("tabla_silver", "transactions_silver")

catalogo = dbutils.widgets.get("catalogo")
tabla_bronze = f"{catalogo}.{dbutils.widgets.get('esquema_bronze')}.{dbutils.widgets.get('tabla_bronze')}"
#tabla_drive = f"{catalogo}.{dbutils.widgets.get('esquema_bronze')}.{dbutils.widgets.get('tabla_drive')}"
tabla_silver = f"{catalogo}.{dbutils.widgets.get('esquema_silver')}.{dbutils.widgets.get('tabla_silver')}"
print("Source 1:", tabla_bronze)
#print("Source 2:", tabla_drive)
print("Target:", tabla_silver)

In [0]:
df_transactions_bronze = spark.table(tabla_bronze)
#df_drive_bronze = spark.table(tabla_drive)

df_bronze = df_transactions_bronze #.unionByName(df_drive_bronze)
print("Transactions rows:", df_transactions_bronze.count())
#print("Drive rows:", df_drive_bronze.count())
print("Combined bronze rows:", df_bronze.count())
display(df_bronze.limit(5))

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Trim and collapse repeated inner spaces in every string column
string_cols = [f.name for f in df_bronze.schema.fields if f.dataType.simpleString() == "string"]

df_clean = df_bronze.select([
    F.regexp_replace(F.trim(F.col(c)), r"\s+", " ").alias(c) if c in string_cols else F.col(c)
    for c in df_bronze.columns
])

# IDs always in upper case
for c in ["transaction_id", "customer_id", "product_id"]:
    df_clean = df_clean.withColumn(c, F.upper(F.col(c)))

In [0]:
# Bronze has many case variants of the same product (e.g. 'pHONE cHARGER', 'PHONE CHARGER').
# Pick one canonical name per product_id: the most frequent "properly cased" variant
# (starts with upper case, not all upper, not all lower). Fallback: initcap.
proper = (
    (F.col("product_name") != F.upper("product_name"))
    & (F.col("product_name") != F.lower("product_name"))
    & (F.substring("product_name", 1, 1) == F.upper(F.substring("product_name", 1, 1)))
)

w = Window.partitionBy("product_id").orderBy(F.desc("is_proper"), F.desc("cnt"), F.asc("product_name"))

dim_product_name = (
    df_clean.groupBy("product_id", "product_name").agg(F.count("*").alias("cnt"))
    .withColumn("is_proper", proper.cast("int"))
    .withColumn("rn", F.row_number().over(w))
    .filter("rn = 1")
    .select(
        "product_id",
        F.when(F.col("is_proper") == 1, F.col("product_name"))
         .otherwise(F.initcap("product_name")).alias("product_name_std"),
    )
)

df_clean = (
    df_clean.join(F.broadcast(dim_product_name), "product_id", "left")
    .withColumn("product_name", F.coalesce("product_name_std", F.initcap("product_name")))
    .drop("product_name_std")
)

display(dim_product_name.orderBy("product_id"))

In [0]:
# 1. Remove duplicate transactions (keep latest date per transaction_id)
w_dup = Window.partitionBy("transaction_id").orderBy(F.desc("transaction_date"))
df_clean = df_clean.withColumn("rn", F.row_number().over(w_dup)).filter("rn = 1").drop("rn")

# 2. Keep only valid records
df_clean = df_clean.filter(
    F.col("transaction_id").isNotNull()
    & F.col("transaction_date").isNotNull()
    & (F.col("quantity") > 0)
    & (F.col("unit_price") > 0)
)

# 3. Recalculate total_amount to guarantee consistency, rounded to 2 decimals
df_clean = df_clean.withColumn("total_amount", F.round(F.col("quantity") * F.col("unit_price"), 2)) \
                   .withColumn("unit_price", F.round("unit_price", 2))

# 4. Derived columns + audit column
df_silver = (
    df_clean
    .withColumn("transaction_day", F.to_date("transaction_date"))
    .withColumn("transaction_hour", F.hour("transaction_date"))
    .withColumn("day_of_week", F.date_format("transaction_date", "EEEE"))
    .withColumn("silver_load_ts", F.current_timestamp())
    .select(
        "transaction_id", "transaction_date", "transaction_day", "transaction_hour", "day_of_week",
        "customer_id", "product_id", "product_name", "category",
        "quantity", "unit_price", "total_amount",
        "store_location", "payment_method", "silver_load_ts",
    )
)

print("Bronze rows:", df_bronze.count(), "| Silver rows:", df_silver.count())
display(df_silver.limit(10))

In [0]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(tabla_silver)

In [0]:
silver = spark.table(tabla_silver)

display(silver.agg(
    F.count("*").alias("rows"),
    F.countDistinct("transaction_id").alias("distinct_ids"),
    F.countDistinct("product_name").alias("distinct_product_names"),
    F.countDistinct("product_id").alias("distinct_product_ids"),
    F.sum(F.col("transaction_date").isNull().cast("int")).alias("null_dates"),
    F.min("transaction_date").alias("min_date"),
    F.max("transaction_date").alias("max_date"),
    F.round(F.sum("total_amount"), 2).alias("total_sales"),
))